# 06 — ETA lúc nhận đơn và đối chiếu tính tay

Protocol B dự đoán khi shipper nhận đơn. Mỗi mẫu ứng với một order_id.
Nhãn là số phút từ nhận đến giao. Ví dụ dùng phiên bản và cửa sổ chấm
mà Prefect công bố.

### 1. Mở đúng file đầu vào

File **data/delivery_yt.csv** nằm trong repo và được mount chỉ đọc tại
**/home/jovyan/work/data/delivery_yt.csv** trong Docker. Cell kiểm tra SHA-256
theo config/prereg.yaml. Prefect đọc dữ liệu từ Bronze. Phần tính tay đọc trực
tiếp CSV bằng kernel Python của môi trường đã cài repo với pip install -e .

In [1]:
import hashlib
from math import asin, cos, fsum, isclose, radians, sin, sqrt

import numpy as np
import pandas as pd
from lakehouse import PROJECT_ROOT as ROOT

from eta.dataset import (
    ACCEPT_VARIANTS, FLOW_ACCEPT_VARIANT, OBSERVED_ACCEPT_VARIANTS, SOURCE_COLUMNS, TARGET,
    build_protocol_b_samples, flow_evaluation_rows, load_accept_prereg, read_lade_csv,
)
from eta.metrics import accept_eta_metrics
from eta.model import fit_accept_eta

prereg = load_accept_prereg()
source_config = prereg["data"]
source = ROOT / source_config["source"]

with source.open("rb") as handle:
    checksum = hashlib.file_digest(handle, "sha256").hexdigest()
assert checksum == source_config["sha256"], f"SHA-256 của {source} sai: {checksum}"

events = read_lade_csv(source)
orders = events.set_index("order_id", verify_integrity=True)
print(f"File cấu hình  {source_config['source']}")
print(f"SHA-256        {checksum}")
print(f"CSV            {len(events):,} dòng × {len(SOURCE_COLUMNS)} cột")
print(events[["order_id", "courier_id", "aoi_id", "accept_time", "delivery_time"]]
      .head(3).to_string(index=False))

File cấu hình  data/delivery_yt.csv
SHA-256        aa5bca7fdaa8c89d4ab50a2ed2b94fb09f608acc49fc1108a1a612e9c0f5a7ad
CSV            206,431 dòng × 17 cột
order_id  courier_id  aoi_id    accept_time  delivery_time
 2071340        3135     175 06-18 08:00:00 06-18 16:21:00
 3767291        3135     175 08-14 08:30:00 08-14 15:01:00
 2184812        3135     175 09-01 09:08:00 09-01 15:15:00


### 2. Lọc nhãn hợp lệ và chia theo thời gian

Builder kiểm tra 17 cột và gắn năm 2022. Để chấm điểm cần đơn có mốc nhận,
mốc giao hợp lệ và thời lượng dương. Từ tập có nhãn, bộ lấy mẫu giữ tối
đa 12 đơn mỗi shipper mỗi ngày bằng seed 42. Hàng chờ xét cả đơn đã nhận
mà thời điểm truy vấn chưa có sự kiện giao hợp lệ.

Cửa sổ cuối của rolling-origin có ba đoạn gồm fit, hiệu chỉnh dừng sớm
và chấm điểm. Nhãn fit phải xuất hiện trước đoạn hiệu chỉnh. Nhãn hiệu
chỉnh phải xuất hiện trước cửa sổ chấm. [Nguồn và các mốc cắt](../docs/eta/training_data_lineage.md)

In [2]:
valid_time = events.accept_ts.notna() & events.delivery_ts.notna()
positive_duration = valid_time & events.delivery_minutes.gt(0)
variant = FLOW_ACCEPT_VARIANT
samples = build_protocol_b_samples(events, source_config, observed=variant in OBSERVED_ACCEPT_VARIANTS)
train, calibration, evaluation, scope = flow_evaluation_rows(samples, orders)

counts = pd.Series({
    "Dòng gốc": len(events),
    "Timestamp hợp lệ và thời lượng dương": int(positive_duration.sum()),
    "Mẫu sau khi chọn tối đa 12 đơn mỗi shipper/ngày": len(samples),
    "Mẫu fit có nhãn kịp thời": len(train),
    "Mẫu hiệu chỉnh dừng sớm": len(calibration),
    "Mẫu chấm trên cửa sổ cuối": len(evaluation),
    "Holdout đã khóa": int(samples.split.eq("test").sum()),
})
print(counts.to_string())
print(f"{variant} · {scope}")
print(f"Cửa sổ chấm từ {evaluation.query_ts.min().date()} đến {evaluation.query_ts.max().date()}")

Dòng gốc                                           206431
Timestamp hợp lệ và thời lượng dương               206062
Mẫu sau khi chọn tối đa 12 đơn mỗi shipper/ngày    123117
Mẫu fit có nhãn kịp thời                            80371
Mẫu hiệu chỉnh dừng sớm                              9766
Mẫu chấm trên cửa sổ cuối                            7610
Holdout đã khóa                                     25345
protocol_b_queue_history · rolling_validation
Cửa sổ chấm từ 2022-09-13 đến 2022-09-24


### 3. Tính nhãn và đặc trưng từ một đơn thật

Chọn một đơn có GPS nhận hợp lệ và tối đa ba đơn cùng AOI đã giao trước
lúc nhận. Đối chiếu từng giá trị với bảng mẫu.

- Nhãn bằng giờ giao trừ giờ nhận.
- Hàng chờ là số đơn của shipper đã nhận trước truy vấn trừ số đơn đã có
  thời điểm giao hợp lệ trước truy vấn.
- Trung bình AOI lấy thời lượng của đơn đã giao. Cửa sổ 28 ngày dùng
  thời điểm giao để xác định lịch sử gần đây.
- Khoảng cách là Haversine giữa GPS được ghi tới lúc nhận và tọa độ đích.

Giờ giao của đơn minh họa dùng để chấm sai số.

In [3]:
candidates = evaluation.loc[evaluation.aoi_past_count.between(1, 3)
                            & evaluation.aoi_recent_count.gt(0)
                            & evaluation.accept_to_stop_km.notna()]
example = candidates.sort_values(["query_ts", "order_id"]).iloc[0]
order = orders.loc[example.order_id]
accepted = pd.Timestamp(f"2022-{order.accept_time}")
delivered = pd.Timestamp(f"2022-{order.delivery_time}")
hand_label = (delivered - accepted).total_seconds() / 60

completed = events.delivery_ts.lt(accepted) & events.delivery_ts.gt(events.accept_ts)
courier = events.courier_id.eq(order.courier_id)
earlier = events.accept_ts.lt(accepted)
hand_pending = int((courier & earlier).sum() - (courier & completed).sum())
history = events.loc[events.aoi_id.eq(order.aoi_id) & completed].sort_values("delivery_ts")
hand_durations = (history.delivery_ts - history.accept_ts).dt.total_seconds() / 60
recent = history.delivery_ts.ge(accepted - pd.Timedelta(days=28))

start_lat, start_lng, stop_lat, stop_lng = map(
    radians, (order.accept_gps_lat, order.accept_gps_lng, order.lat, order.lng),
)
a = sin((stop_lat - start_lat) / 2)**2 + cos(start_lat) * cos(stop_lat) * sin((stop_lng - start_lng) / 2)**2
hand_distance = 2 * 6371.0088 * asin(sqrt(a))
manual = pd.Series({
    TARGET: hand_label,
    "pending_order_count": hand_pending,
    "aoi_past_count": len(history),
    "aoi_past_avg_min": hand_durations.mean(),
    "aoi_recent_count": int(recent.sum()),
    "aoi_recent_avg_min": hand_durations.loc[recent].mean(),
    "accept_to_stop_km": hand_distance,
})
feature_check = pd.DataFrame({"Tính tay": manual, "Builder": example[manual.index].astype(float)})
np.testing.assert_allclose(feature_check["Tính tay"], feature_check["Builder"], rtol=0, atol=1e-9)

print(f"Đơn {example.order_id}, AOI {order.aoi_id}")
print(f"Nhãn = {order.delivery_time} - {order.accept_time} = {hand_label:.0f} phút")
print(f"Hàng chờ = {(courier & earlier).sum()} đã nhận - {(courier & completed).sum()} đã giao = {hand_pending}")
print(history[["order_id", "accept_time", "delivery_time"]]
      .assign(duration_min=hand_durations).to_string(index=False))
print(feature_check.to_string(float_format=lambda value: f"{value:.8f}"))

Đơn 1818091, AOI 9093
Nhãn = 09-13 21:05:00 - 09-13 07:48:00 = 797 phút
Hàng chờ = 209 đã nhận - 206 đã giao = 3
order_id    accept_time  delivery_time  duration_min
 3540372 08-29 08:02:00 08-29 20:49:00         767.0
                             Tính tay      Builder
target_remaining_minutes 797.00000000 797.00000000
pending_order_count        3.00000000   3.00000000
aoi_past_count             1.00000000   1.00000000
aoi_past_avg_min         767.00000000 767.00000000
aoi_recent_count           1.00000000   1.00000000
aoi_recent_avg_min       767.00000000 767.00000000
accept_to_stop_km         25.79892085  25.79892085


### 4. Đưa đúng vector đặc trưng vào LightGBM

Notebook và benchmark dùng cùng fit_accept_eta(). Giá trị thiếu lấy median
từ tập fit. LightGBM chọn số cây bằng đoạn hiệu chỉnh đứng trước đoạn chấm.
Hàng in dưới đây là vector đưa vào bộ dự đoán cho đơn đang kiểm tra.

In [4]:
features = ACCEPT_VARIANTS[variant]
seed = prereg["model"]["seeds"][0]
fitted = fit_accept_eta(train, calibration, features, seed)
model = fitted.estimator
model_input = fitted.inputs(evaluation.loc[[example.name]])
print(model_input.T.rename(columns={example.name: "Giá trị đưa vào mô hình"}).to_string())
print(f"Seed {seed}, {model.best_iteration_} cây ở lần dừng tốt nhất.")

                             Giá trị đưa vào mô hình
pending_order_count                                3
completed_order_count                            206
courier_avg_min                                669.5
distance_to_last_delivery_km               39.675116
candidate_lat                               36.72564
candidate_lng                              120.66925
candidate_aoi_type                                14
candidate_region_id                              164
accept_hour                                      7.8
aoi_past_avg_min                               767.0
aoi_past_count                                     1
accept_to_stop_km                          25.798921
courier_accepted_today                             1
courier_delivered_today                            0
courier_idle_min                               613.0
aoi_recent_count                                   1
aoi_recent_avg_min                             767.0
region_hour_recent_count                      

### 5. Lần theo các cây rồi cộng để kiểm tra dự đoán

Mỗi nút so một đặc trưng với ngưỡng. Đi trái nếu giá trị nhỏ hơn hoặc bằng
ngưỡng. Đi phải nếu lớn hơn. Khi tới lá, lấy `leaf_value` đã lưu trong cây.

Cộng ba giá trị lá đầu để so với `predict(num_iteration=3)` của cùng model.
Sau đó lặp phép cộng cho toàn bộ cây và chặn dưới ở 0 phút theo benchmark.

In [5]:
def follow_tree(node, row, record_path=False):
    path = []
    while "split_index" in node:
        assert node["decision_type"] == "<=" and node["missing_type"] != "Zero"
        column = features[node["split_feature"]]
        value = float(row[column])
        threshold = float(node["threshold"])
        child = "left_child" if value <= threshold else "right_child"
        if record_path:
            branch = "trái" if child == "left_child" else "phải"
            path.append(f"{column} ({value!r}) <= {threshold!r} → {branch}")
        node = node[child]
    return float(node["leaf_value"]), path

dump = model.booster_.dump_model(num_iteration=model.best_iteration_)
assert dump["num_tree_per_iteration"] == 1 and not dump["average_output"]
assert model_input.notna().all().all()
row = model_input.iloc[0]
leaves = []
for index, tree in enumerate(dump["tree_info"]):
    value, path = follow_tree(tree["tree_structure"], row, record_path=index < 3)
    leaves.append(value)
    if index < 3:
        print(f"Cây {index + 1}")
        print("\n".join(f"  {decision}" for decision in path))
        print(f"  Lá {value:.10f}")

hand_three = fsum(leaves[:3])
machine_three = float(model.predict(model_input, num_iteration=3)[0])
hand_full = max(0.0, fsum(leaves))
machine_full = float(fitted.predict(evaluation.loc[[example.name]])[0])
assert isclose(hand_three, machine_three, rel_tol=0, abs_tol=1e-9)
assert isclose(hand_full, machine_full, rel_tol=0, abs_tol=1e-9)

summands = " + ".join(f"({value:.10f})" for value in leaves[:3])
print(f"Ba cây = {summands} = {hand_three:.10f} phút")
print(f"Máy dùng ba cây      {machine_three:.10f} phút")
print(f"Cộng đủ {len(leaves)} cây    {hand_full:.10f} phút")
print(f"Máy dùng đủ cây       {machine_full:.10f} phút")
print(f"Nhãn thực tế {hand_label:.2f} phút, sai số tuyệt đối {abs(hand_full - hand_label):.2f} phút")

Cây 1
  aoi_recent_avg_min (767.0) <= 208.79743589743592 → phải
  accept_hour (7.8) <= 12.29166666666667 → trái
  aoi_recent_avg_min (767.0) <= 275.03125000000006 → phải
  courier_recent_avg_min (542.8157894736842) <= 186.53982029814173 → phải
  aoi_recent_avg_min (767.0) <= 351.40731707317076 → phải
  candidate_lng (120.66925) <= 119.97205000000001 → phải
  Lá 204.9600000000
Cây 2
  aoi_recent_avg_min (767.0) <= 208.79743589743592 → phải
  aoi_past_avg_min (767.0) <= 271.5023364485982 → phải
  accept_hour (7.8) <= 13.941666666666668 → trái
  courier_avg_min (669.5) <= 194.61164767552978 → phải
  aoi_past_avg_min (767.0) <= 324.7464285714286 → phải
  courier_recent_gap_avg_min (20.548387096774192) <= 29.82774657774658 → trái
  Lá 6.5412000000
Cây 3
  aoi_recent_avg_min (767.0) <= 208.79743589743592 → phải
  aoi_past_avg_min (767.0) <= 271.5023364485982 → phải
  courier_delivered_today (0.0) <= 1.5000000000000002 → trái
  courier_avg_min (669.5) <= 190.01987819436997 → phải
  aoi_past_a

### 6. Chấm cửa sổ validation cuối

Đối chiếu một đơn kiểm tra phép tính của mô hình. MAE, WAPE, ACC@30 và
ACC@60 đo sai số trên các đơn ở cửa sổ chấm. Bảng rolling-origin tổng hợp
cả ba cửa sổ và cho biết biến thể nào nhất quán theo thời gian.
[Benchmark trên cùng tập đơn](../reports/eta/eta_accept_benchmark_rolling.csv)

In [6]:
prediction = fitted.predict(evaluation)
metrics = accept_eta_metrics(evaluation[TARGET].to_numpy(), prediction)
print(f"{variant}, seed {seed}, {len(evaluation):,} đơn")
print(pd.Series(metrics).drop("rmse").to_string(float_format=lambda value: f"{value:.8f}"))

protocol_b_queue_history, seed 42, 7,610 đơn
n                   7610.00000000
mae                  107.25274975
acc20_pct             21.98423127
acc30_pct             32.07621551
acc60_pct             56.01839685
p90_abs_error_min    217.76664521
wape_pct              39.65474661


### 7. Khoảng ETA và hạn hứa P90

Ba mô hình quantile dùng cùng đặc trưng và cùng ba cửa sổ với benchmark Protocol B. Các ngày hiệu chỉnh được chia theo thời gian cho dừng sớm và conformal. Điểm hiệu chỉnh khoảng 80% và điểm hiệu chỉnh hạn hứa P90 được tính riêng.

Cell dưới đọc bảng kết quả đã lưu. Để đối chiếu một đơn thật, chạy lệnh sau tại thư mục dự án để sinh CSV theo đơn từ `data/delivery_yt.csv`.

~~~bash
python -m scripts.experiments.run_eta_accept_benchmark --intervals --seeds 42 43 44 --deadline-minutes 360 --predictions-output reports/eta/eta_predictions_intervals.csv
~~~

Từ P10/P50/P90 và hai điểm conformal của cửa sổ, tính lại cận dưới, cận trên và hạn hứa bằng số phút từ lúc nhận. Cờ nguy cơ trễ bật khi hạn hứa vượt mốc 360 phút minh họa. [Phương pháp và giới hạn](../docs/eta/analysis_limits.md#7-khoảng-eta-lúc-nhận-đơn)

In [ ]:
interval_dir = ROOT / "reports" / "eta"
interval_report = pd.read_csv(interval_dir / "eta_accept_intervals_rolling.csv")
summary_columns = ["variant", "seed", "n", "interval_coverage_pct", "median_width_min",
                   "promise_coverage_pct", "median_promise_min"]
print(interval_report.loc[interval_report.scope.eq("pooled"), summary_columns].round(2).to_string(index=False))

per_order_path = interval_dir / "eta_predictions_intervals.csv"
if per_order_path.is_file():
    per_order = pd.read_csv(per_order_path, dtype={"order_id": str})
    candidates = per_order.loc[per_order.variant.eq(FLOW_ACCEPT_VARIANT)]
    if "late_risk" in candidates:
        candidates = candidates.sort_values("late_risk", ascending=False, kind="stable")
    example = candidates.iloc[0]
    fit_window = interval_report.loc[
        interval_report.scope.eq("window") & interval_report.variant.eq(example.variant)
        & interval_report.seed.eq(example.seed) & interval_report.group.astype(str).eq(str(example.window))
    ].iloc[0]
    p10, p50, p90 = example[["p10_minutes", "p50_minutes", "p90_minutes"]]
    interval_offset, promise_offset = fit_window[["interval_offset_min", "promise_offset_min"]]
    lower = min(p50, max(0, p10 - interval_offset))
    upper = max(p50, p90 + interval_offset)
    promise = max(p50, p90 + promise_offset)
    np.testing.assert_allclose([lower, upper, promise],
                               example[["lower_80_minutes", "upper_80_minutes", "promise_p90_minutes"]].astype(float),
                               rtol=0, atol=1e-6)

    raw_order = orders.loc[example.order_id]
    actual_minutes = (raw_order.delivery_ts - raw_order.accept_ts).total_seconds() / 60
    assert isclose(actual_minutes, example[TARGET], abs_tol=1e-6)
    print(f"P10/P50/P90 gốc {p10:.4f} / {p50:.4f} / {p90:.4f} phút")
    print(f"Điểm hiệu chỉnh khoảng 80% {interval_offset:.4f} · hạn hứa P90 {promise_offset:.4f} phút")
    print(f"Đơn {example.order_id} · nhãn {actual_minutes:.2f} phút · khoảng [{lower:.2f}, {upper:.2f}] · hứa P90 {promise:.2f} phút")
    if "late_risk" in example:
        late_risk = promise > example.deadline_minutes
        assert late_risk == example.late_risk
        print(f"Hạn minh họa {example.deadline_minutes:.0f} phút · nguy cơ trễ {late_risk}")
else:
    print("Chạy lệnh trên để sinh CSV đối chiếu theo đơn.")

### Ghi kết quả của bước ETA vào Gold và MySQL

Prefect gọi build_eta_safe() bên dưới. Kết quả lưu mã biến thể, seed và
mốc của cửa sổ validation để dashboard hiển thị đúng phạm vi điểm số.
Cell ghi bảng riêng dùng cho môi trường Docker.

In [ ]:
def build_eta_safe(spark, seeds=None, *, run_id=None, variant=None):
    """Fit and score the accept-time model in a development window."""
    from datetime import timedelta
    import uuid

    from eta.dataset import (
        ACCEPT_VARIANTS, FLOW_ACCEPT_VARIANT, OBSERVED_ACCEPT_VARIANTS, TARGET,
        build_protocol_b_samples, flow_evaluation_rows, load_accept_prereg, parse_lade_frame,
    )
    from eta.metrics import accept_eta_metrics
    from eta.model import fit_accept_eta

    prereg = load_accept_prereg()
    seeds = prereg["model"]["seeds"] if seeds is None else seeds
    if not seeds:
        raise ValueError("At least one model seed is required")
    variant = variant or FLOW_ACCEPT_VARIANT
    features = ACCEPT_VARIANTS[variant]

    raw = spark.read.format("delta").load(BRONZE).toPandas()
    events = parse_lade_frame(raw)
    orders = events.set_index("order_id", verify_integrity=True)
    samples = build_protocol_b_samples(events, prereg["data"], observed=variant in OBSERVED_ACCEPT_VARIANTS)
    train, calibration, evaluation, scope = flow_evaluation_rows(samples, orders)
    score_start = evaluation.query_ts.min().date().isoformat()
    score_end = (evaluation.query_ts.max().date() + timedelta(days=1)).isoformat()
    active_run_id = run_id or uuid.uuid4().hex
    scored = evaluation[["order_id", "courier_id", "query_ts", "courier_day", TARGET]].copy()
    scored["run_id"] = active_run_id
    scored["variant"] = variant
    scored["split"] = "validation"
    scored["evaluation_scope"] = scope
    scored["seed"] = seeds[0]

    actual = evaluation[TARGET].to_numpy()
    metrics = []
    for seed in seeds:
        prediction = fit_accept_eta(train, calibration, features, seed).predict(evaluation)
        result = accept_eta_metrics(actual, prediction)
        metrics.append((
            active_run_id, "B", "validation", result["n"], result["mae"], result["rmse"],
            seed, "bronze_to_pandas", variant, scope, score_start, score_end,
            result["wape_pct"], result["acc30_pct"], result["acc60_pct"], result["p90_abs_error_min"],
        ))
        if seed == seeds[0]:
            scored["prediction_minutes"] = prediction

    spark.createDataFrame(scored).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ETA_SAFE)
    metrics_schema = (
        "run_id string, protocol string, split string, n long, mae double, rmse double, seed int, execution_mode string, "
        "variant string, evaluation_scope string, score_start string, score_end_exclusive string, "
        "wape_pct double, acc30_pct double, acc60_pct double, p90_abs_error_min double"
    )
    metrics_frame = spark.createDataFrame(metrics, metrics_schema)
    metrics_frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ETA_SAFE_METRICS)
    write_mysql(metrics_frame, "eta_safe_metrics", run_id=run_id)
    return {
        "run_id": active_run_id, "variant": variant, "evaluation_scope": scope,
        "score_start": score_start, "score_end_exclusive": score_end,
        "train_rows": len(train), "calibration_rows": len(calibration), "validation_rows": len(evaluation),
        "validation_mae": metrics[0][4], "seeds": list(seeds),
    }

In [ ]:
RUN_PUBLISH_ACCEPT_ETA = False
if RUN_PUBLISH_ACCEPT_ETA:
    from lakehouse.notebooks import load_notebook_steps

    steps = load_notebook_steps()
    spark = steps.get_spark("Delivery - Protocol B")
    print(steps.build_eta_safe(spark))

<details>
<summary>Kết quả B đăng ký đã lưu từ lần chạy trước</summary>

Lần chạy lịch sử dưới đây chấm Protocol B gốc trên test. Khi đó flow fit cả 73.125 nhãn train. Kết quả giữ để truy vết lần chạy đã lưu.

~~~text
{'run_id': 'ee33b1bcc0864b3db7fd8937600d8aa0',
 'train_rows': 73125,
 'validation_rows': 24647,
 'test_rows': 25345,
 'test_mae': 117.45598968125002,
 'test_rmse': 268.02749142476677,
 'seeds': [42, 43, 44]}
~~~

</details>